# Robosuite Scripted Data Collection — All 6 Scenarios

This notebook collects demonstration data using **scripted policies** for 6 robosuite tasks:

| # | Environment | Description | Target Demos |
|---|---|---|---|
| 1 | **Lift** | Pick up a cube and lift it | 50 |
| 2 | **Stack** | Pick red cube, stack on green cube | 50 |
| 3 | **PickPlaceSingle** | Pick object, place in target bin | 50 |
| 4 | **NutAssemblySquare** | Place square nut on square peg | 75 |
| 5 | **NutAssemblyRound** | Place round nut on round peg | 75 |
| 6 | **NutAssembly** | Both nuts on their pegs | 75 |

## Pipeline
1. **Setup** — Install deps, configure EGL rendering
2. **Scripted Policies** — Proper hover→descend→grip→lift→move→place sequences
3. **Trial Runs** — 2 episodes per scenario with video output (verify visually)
4. **Full Collection** — Remaining demos to reach targets, saved as HDF5

**Requirements:** GPU runtime (Runtime → Change runtime type → T4 GPU)

---
## 1. System Setup & Dependencies

In [ ]:
%%bash
# Install system dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# Create NVIDIA EGL ICD config (Colab is missing this by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
# Install Python packages
!pip install -q robosuite imageio[ffmpeg] matplotlib h5py Pillow
!pip install -q "numpy>=2.0,<2.1"

print("\nPackages installed. Restarting runtime to fix numpy C bindings...")
print("After restart, SKIP this cell and continue from the next section.")

import os
os.kill(os.getpid(), 9)

### After runtime restart — continue from here

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

import numpy as np
import robosuite as suite
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import base64
import h5py
import time

print(f"robosuite {robosuite.__version__}")
print(f"numpy {np.__version__}")
print("Setup complete.")

---
## 2. Scripted Policies

Each policy uses a **state-machine** approach with proper phases:
1. **Hover** — move above the target object (open gripper)
2. **Descend** — lower onto the object
3. **Grasp** — close gripper and wait for it to firmly grip
4. **Lift** — raise the object
5. **Move** — transport to target location (for place/stack tasks)
6. **Place/Release** — open gripper at target

All policies use **proportional control** with gain tuning and explicit phase transitions based on distance thresholds.

In [ ]:
# ============================================================
# SCRIPTED POLICIES — All 6 Scenarios
# ============================================================
#
# Key constants:
#   Table height:    0.8m
#   Hover height:    ~0.15m above object
#   Grasp threshold: 0.01m from object center
#   Gripper:         +1 = close, -1 = open (robosuite Panda convention)
#
# Action space (7D OSC_POSE for Panda):
#   [dx, dy, dz, dax, day, daz, gripper]
#   Position deltas are clipped to [-1, 1]
# ============================================================

TABLE_HEIGHT = 0.8
HOVER_DELTA_Z = 0.12     # hover this far above the object
GRASP_XY_THRESH = 0.01   # close enough in XY to descend
GRASP_Z_THRESH = 0.005   # close enough in Z to start gripping
GRIP_WAIT_STEPS = 15     # steps to wait while gripper closes
GAIN = 8.0               # proportional gain for position control


class LiftPolicy:
    """Lift: hover above cube → descend → grip → lift.

    Success condition: cube_z > table_height + 0.04
    Obs keys: cube_pos, robot0_eef_pos
    """
    def __init__(self):
        self.phase = "hover"  # hover → descend → grip → lift
        self.grip_counter = 0

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0

    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        cube = obs["cube_pos"]
        action = np.zeros(7)

        if self.phase == "hover":
            # Move above cube with gripper open
            target = cube.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            # Lower onto cube
            target = cube.copy()
            target[2] += 0.003  # slightly above cube center for top-grasp
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # keep open
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            # Close gripper and wait
            target = cube.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN * 0.5  # gentle position hold
            action[6] = 1  # close
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            # Lift straight up
            action[2] = 1.0  # max upward velocity
            action[6] = 1    # keep closed

        return np.clip(action, -1, 1)


class StackPolicy:
    """Stack: pick cubeA (red) → place on cubeB (green).

    Success condition: cubeA on cubeB, not grasped, cubeA touching cubeB
    Obs keys: cubeA_pos, cubeB_pos, robot0_eef_pos
    """
    def __init__(self):
        self.phase = "hover_A"  # hover_A → descend_A → grip_A → lift_A → hover_B → descend_B → release
        self.grip_counter = 0
        self.release_counter = 0

    def reset(self):
        self.phase = "hover_A"
        self.grip_counter = 0
        self.release_counter = 0

    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        cubeA = obs["cubeA_pos"]
        cubeB = obs["cubeB_pos"]
        action = np.zeros(7)

        if self.phase == "hover_A":
            target = cubeA.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend_A"

        elif self.phase == "descend_A":
            target = cubeA.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip_A"
                self.grip_counter = 0

        elif self.phase == "grip_A":
            target = cubeA.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1  # close
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift_A"

        elif self.phase == "lift_A":
            # Lift cubeA high enough to clear cubeB
            lift_target = cubeB.copy()
            lift_target[2] += 0.20  # well above cubeB
            delta = lift_target - ee
            action[2] = 1.0  # strong upward
            action[6] = 1    # keep gripping
            if ee[2] > cubeB[2] + 0.15:
                self.phase = "hover_B"

        elif self.phase == "hover_B":
            # Move above cubeB while holding cubeA
            target = cubeB.copy()
            target[2] += 0.10  # above cubeB
            delta = target - ee
            action[:3] = delta * GAIN * 0.6  # gentler to not drop
            action[6] = 1  # keep gripping
            if np.linalg.norm(delta[:2]) < 0.015 and abs(delta[2]) < 0.02:
                self.phase = "descend_B"

        elif self.phase == "descend_B":
            # Lower cubeA onto cubeB
            target = cubeB.copy()
            # cubeB is 0.025m half-size, cubeA is 0.02m half-size
            # We want cubeA sitting on top of cubeB
            target[2] += 0.05  # cubeB_height + cubeA_half
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1  # keep gripping
            if np.linalg.norm(delta) < 0.015:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            # Open gripper and back away
            action[6] = -1  # open
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0  # back away upward

        return np.clip(action, -1, 1)


class PickPlaceSinglePolicy:
    """PickPlaceSingle: pick object from table → place in bin2.

    Success condition: object in target bin AND gripper not too close.
    Obs keys: We detect the active object by checking known object names.
    Bin2 (target) is at approximately (0.1, 0.28, 0.8).
    """
    OBJ_NAMES = ["Milk", "Bread", "Cereal", "Can"]

    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.active_obj = None

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.active_obj = None

    def _find_object(self, obs):
        """Find which object is active by checking obs keys."""
        if self.active_obj is not None:
            return self.active_obj
        for name in self.OBJ_NAMES:
            if f"{name}_pos" in obs:
                self.active_obj = name
                return name
        return None

    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        obj_name = self._find_object(obs)
        if obj_name is None:
            return np.zeros(7)
        obj_pos = obs[f"{obj_name}_pos"]
        # Target bin2 position
        bin2 = np.array([0.1, 0.28, 0.8])

        action = np.zeros(7)

        if self.phase == "hover":
            target = obj_pos.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = obj_pos.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = obj_pos.copy()
            target[2] += 0.003
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1  # close
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > bin2[2] + 0.20:
                self.phase = "move_to_bin"

        elif self.phase == "move_to_bin":
            target = bin2.copy()
            target[2] = ee[2]  # maintain height while moving horizontally
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1  # keep gripping
            if np.linalg.norm(delta[:2]) < 0.03:
                self.phase = "lower_to_bin"

        elif self.phase == "lower_to_bin":
            target = bin2.copy()
            target[2] += 0.10  # above bin
            delta = target - ee
            action[:3] = delta * GAIN * 0.4
            action[6] = 1
            if abs(delta[2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1  # open
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0  # back away

        return np.clip(action, -1, 1)


class NutAssemblySquarePolicy:
    """NutAssemblySquare: pick square nut → place on square peg.

    Success: nut within 0.03m (xy) of peg and z < table+0.05.
    Obs keys: SquareNut_pos, robot0_eef_pos.
    The peg position is not directly in obs, so we use the env handle
    to read self.sim.data.body_xpos[self.peg1_body_id].
    We'll pass the peg position to the policy from the run loop.
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def __call__(self, obs, peg_pos=None):
        ee = obs["robot0_eef_pos"]
        nut = obs["SquareNut_pos"]
        action = np.zeros(7)

        if peg_pos is None:
            peg_pos = np.array([0.12, 0.12, 0.8])  # fallback

        if self.phase == "hover":
            target = nut.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1  # open
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1  # close
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            target = peg_pos.copy()
            target[2] = ee[2]  # maintain height
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "lower_to_peg"

        elif self.phase == "lower_to_peg":
            target = peg_pos.copy()
            target[2] += 0.04  # just above peg
            delta = target - ee
            action[:3] = delta * GAIN * 0.3
            action[6] = 1
            if abs(delta[2]) < 0.02 and np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1  # open
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0  # back away

        return np.clip(action, -1, 1)


class NutAssemblyRoundPolicy:
    """NutAssemblyRound: pick round nut → place on round peg.

    Obs keys: RoundNut_pos, robot0_eef_pos.
    Peg2 position read from env.
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0

    def __call__(self, obs, peg_pos=None):
        ee = obs["robot0_eef_pos"]
        nut = obs["RoundNut_pos"]
        action = np.zeros(7)

        if peg_pos is None:
            peg_pos = np.array([0.12, -0.12, 0.8])  # fallback

        if self.phase == "hover":
            target = nut.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            target = peg_pos.copy()
            target[2] = ee[2]
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "lower_to_peg"

        elif self.phase == "lower_to_peg":
            target = peg_pos.copy()
            target[2] += 0.04
            delta = target - ee
            action[:3] = delta * GAIN * 0.3
            action[6] = 1
            if abs(delta[2]) < 0.02 and np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 10:
                action[2] = 1.0

        return np.clip(action, -1, 1)


class NutAssemblyFullPolicy:
    """NutAssembly (both): pick square nut → peg1, then round nut → peg2.

    Success: BOTH nuts on their pegs.
    Obs keys: SquareNut_pos, RoundNut_pos, robot0_eef_pos.
    """
    def __init__(self):
        self.current_nut = "square"  # do square first, then round
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.retreat_counter = 0

    def reset(self):
        self.current_nut = "square"
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.retreat_counter = 0

    def __call__(self, obs, peg1_pos=None, peg2_pos=None):
        ee = obs["robot0_eef_pos"]
        action = np.zeros(7)

        if self.current_nut == "square":
            nut = obs["SquareNut_pos"]
            peg = peg1_pos if peg1_pos is not None else np.array([0.12, 0.12, 0.8])
        else:
            nut = obs["RoundNut_pos"]
            peg = peg2_pos if peg2_pos is not None else np.array([0.12, -0.12, 0.8])

        if self.phase == "hover":
            target = nut.copy()
            target[2] += HOVER_DELTA_Z
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"

        elif self.phase == "descend":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN
            action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"
                self.grip_counter = 0

        elif self.phase == "grip":
            target = nut.copy()
            target[2] += 0.005
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS:
                self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0
            action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            target = peg.copy()
            target[2] = ee[2]
            delta = target - ee
            action[:3] = delta * GAIN * 0.5
            action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "lower_to_peg"

        elif self.phase == "lower_to_peg":
            target = peg.copy()
            target[2] += 0.04
            delta = target - ee
            action[:3] = delta * GAIN * 0.3
            action[6] = 1
            if abs(delta[2]) < 0.02 and np.linalg.norm(delta[:2]) < 0.02:
                self.phase = "release"
                self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1
            self.release_counter += 1
            if self.release_counter > 15:
                self.phase = "retreat"
                self.retreat_counter = 0

        elif self.phase == "retreat":
            # Move up and away before switching to next nut
            action[2] = 1.0
            action[6] = -1
            self.retreat_counter += 1
            if self.retreat_counter > 20:
                if self.current_nut == "square":
                    # Switch to round nut
                    self.current_nut = "round"
                    self.phase = "hover"
                    self.grip_counter = 0
                    self.release_counter = 0
                    self.retreat_counter = 0
                else:
                    # Both done, just hold
                    self.phase = "done"

        elif self.phase == "done":
            pass  # do nothing

        return np.clip(action, -1, 1)


print("All 6 scripted policies defined.")

---
## 3. Environment Factory & Episode Runner

In [ ]:
# ============================================================
# Environment creation and episode execution
# ============================================================

SCENARIOS = {
    "Lift": {
        "env_kwargs": dict(
            env_name="Lift",
            robots="Panda",
            has_renderer=False,
            has_offscreen_renderer=True,
            use_camera_obs=True,
            use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256,
            camera_widths=256,
            reward_shaping=True,
            horizon=400,
        ),
        "policy_class": LiftPolicy,
        "max_steps": 400,
        "target_demos": 50,
        "peg_reader": None,
    },
    "Stack": {
        "env_kwargs": dict(
            env_name="Stack",
            robots="Panda",
            has_renderer=False,
            has_offscreen_renderer=True,
            use_camera_obs=True,
            use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256,
            camera_widths=256,
            reward_shaping=True,
            horizon=500,
        ),
        "policy_class": StackPolicy,
        "max_steps": 500,
        "target_demos": 50,
        "peg_reader": None,
    },
    "PickPlaceSingle": {
        "env_kwargs": dict(
            env_name="PickPlaceSingle",
            robots="Panda",
            has_renderer=False,
            has_offscreen_renderer=True,
            use_camera_obs=True,
            use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256,
            camera_widths=256,
            reward_shaping=True,
            horizon=500,
        ),
        "policy_class": PickPlaceSinglePolicy,
        "max_steps": 500,
        "target_demos": 50,
        "peg_reader": None,
    },
    "NutAssemblySquare": {
        "env_kwargs": dict(
            env_name="NutAssemblySquare",
            robots="Panda",
            has_renderer=False,
            has_offscreen_renderer=True,
            use_camera_obs=True,
            use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256,
            camera_widths=256,
            reward_shaping=True,
            horizon=600,
        ),
        "policy_class": NutAssemblySquarePolicy,
        "max_steps": 600,
        "target_demos": 75,
        "peg_reader": lambda env: env.sim.data.body_xpos[env.peg1_body_id].copy(),
    },
    "NutAssemblyRound": {
        "env_kwargs": dict(
            env_name="NutAssemblyRound",
            robots="Panda",
            has_renderer=False,
            has_offscreen_renderer=True,
            use_camera_obs=True,
            use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256,
            camera_widths=256,
            reward_shaping=True,
            horizon=600,
        ),
        "policy_class": NutAssemblyRoundPolicy,
        "max_steps": 600,
        "target_demos": 75,
        "peg_reader": lambda env: env.sim.data.body_xpos[env.peg2_body_id].copy(),
    },
    "NutAssembly": {
        "env_kwargs": dict(
            env_name="NutAssembly",
            robots="Panda",
            has_renderer=False,
            has_offscreen_renderer=True,
            use_camera_obs=True,
            use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256,
            camera_widths=256,
            reward_shaping=True,
            single_object_mode=0,
            horizon=800,
        ),
        "policy_class": NutAssemblyFullPolicy,
        "max_steps": 800,
        "target_demos": 75,
        "peg_reader": "both",  # special case: reads both pegs
    },
}


def run_episode(env, policy, scenario_name, max_steps, record_video=False):
    """Run one episode. Returns dict with success, frames, actions, etc."""
    obs = env.reset()
    policy.reset()

    frames = []
    actions_buf = []
    obs_buf = []
    total_reward = 0.0
    success = False

    cfg = SCENARIOS[scenario_name]

    for step in range(max_steps):
        # Get peg positions for nut assembly tasks
        if scenario_name == "NutAssemblySquare":
            action = policy(obs, peg_pos=cfg["peg_reader"](env))
        elif scenario_name == "NutAssemblyRound":
            action = policy(obs, peg_pos=cfg["peg_reader"](env))
        elif scenario_name == "NutAssembly":
            peg1 = env.sim.data.body_xpos[env.peg1_body_id].copy()
            peg2 = env.sim.data.body_xpos[env.peg2_body_id].copy()
            action = policy(obs, peg1_pos=peg1, peg2_pos=peg2)
        else:
            action = policy(obs)

        actions_buf.append(action.copy())

        # Store observation data for HDF5
        obs_entry = {}
        if "agentview_image" in obs:
            obs_entry["agentview_image"] = obs["agentview_image"].copy()
        if "robot0_eye_in_hand_image" in obs:
            obs_entry["robot0_eye_in_hand_image"] = obs["robot0_eye_in_hand_image"].copy()
        if "robot0_eef_pos" in obs:
            obs_entry["robot0_eef_pos"] = obs["robot0_eef_pos"].copy()
        if "robot0_eef_quat" in obs:
            obs_entry["robot0_eef_quat"] = obs["robot0_eef_quat"].copy()
        if "robot0_gripper_qpos" in obs:
            obs_entry["robot0_gripper_qpos"] = obs["robot0_gripper_qpos"].copy()
        obs_buf.append(obs_entry)

        # Record frame for video
        if record_video and "agentview_image" in obs:
            frames.append(np.flip(obs["agentview_image"], axis=0).copy())

        obs, reward, done, info = env.step(action)
        total_reward += reward

        # Ground truth success check
        if env._check_success():
            success = True

        if done:
            break

    # Capture final frame
    if record_video and "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0).copy())

    return {
        "success": success,
        "total_reward": total_reward,
        "steps": len(actions_buf),
        "frames": frames,
        "actions": actions_buf,
        "observations": obs_buf,
    }


def save_video_file(frames, path, fps=20):
    """Save frames to MP4."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        writer.append_data(frame)
    writer.close()


def show_video_inline(path):
    """Display MP4 inline in Colab."""
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="512">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
        f'</video>'
    ))


print(f"Configured {len(SCENARIOS)} scenarios.")
for name, cfg in SCENARIOS.items():
    print(f"  {name}: max_steps={cfg['max_steps']}, target_demos={cfg['target_demos']}")

---
## 4. TRIAL RUNS — 2 Episodes Per Scenario (with Video)

Run 2 episodes for each scenario to visually verify the scripted policies work.
**Check the videos before proceeding to full collection.**

In [ ]:
TRIAL_EPISODES = 2
trial_results = {}

for scenario_name, cfg in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"  TRIAL: {scenario_name} ({TRIAL_EPISODES} episodes)")
    print(f"{'='*60}")

    env = suite.make(**cfg["env_kwargs"])
    policy = cfg["policy_class"]()
    results = []

    for ep in range(TRIAL_EPISODES):
        np.random.seed(42 + ep)
        result = run_episode(
            env, policy, scenario_name,
            max_steps=cfg["max_steps"],
            record_video=True,
        )
        results.append(result)

        status = "SUCCESS" if result["success"] else "FAIL"
        print(f"  Episode {ep+1}: {status}  (reward={result['total_reward']:.2f}, steps={result['steps']})")

        # Save and display video
        video_path = f"trial_videos/{scenario_name}_ep{ep+1}.mp4"
        if result["frames"]:
            save_video_file(result["frames"], video_path)
            color = "green" if result["success"] else "red"
            display(HTML(f'<h4 style="color: {color}">{scenario_name} — Episode {ep+1} — {status}</h4>'))
            show_video_inline(video_path)

    env.close()

    n_success = sum(1 for r in results if r["success"])
    trial_results[scenario_name] = {
        "success": n_success,
        "total": TRIAL_EPISODES,
        "rate": n_success / TRIAL_EPISODES,
    }

# Summary
print(f"\n\n{'='*60}")
print(f"  TRIAL SUMMARY")
print(f"{'='*60}")
for name, r in trial_results.items():
    status = "PASS" if r["success"] > 0 else "FAIL"
    print(f"  {name:25s}  {r['success']}/{r['total']}  ({r['rate']:.0%})  [{status}]")

---
## 5. CHECK TRIAL RESULTS

**Review the videos above before proceeding.**

If any scenario shows 0% success rate, the scripted policy needs tuning for that
scenario. If all look good, continue to full data collection below.

In [ ]:
# Quick pass/fail check
all_pass = all(r["success"] > 0 for r in trial_results.values())
if all_pass:
    print("All scenarios had at least 1 success in trial! Safe to proceed.")
else:
    failed = [name for name, r in trial_results.items() if r["success"] == 0]
    print(f"WARNING: These scenarios had 0% trial success: {failed}")
    print("Review the videos above. You may still proceed — the full run with")
    print("more episodes and varied seeds may yield some successes.")

---
## 6. FULL DATA COLLECTION

Collect successful demonstrations and save to HDF5.
- **Lift, Stack, PickPlaceSingle**: 50 successful demos each
- **NutAssemblySquare, NutAssemblyRound, NutAssembly**: 75 successful demos each

The collection loop keeps running episodes until it reaches the target number
of successes, up to a maximum attempt multiplier (3x target).

In [ ]:
import time

OUTPUT_DIR = "collected_demos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Subtract trial successes already collected
# (trial runs used seeds 42, 43 — full run starts from seed 100)
SEED_START = 100
MAX_ATTEMPT_MULTIPLIER = 3  # try up to 3x target demos

collection_summary = {}

for scenario_name, cfg in SCENARIOS.items():
    target = cfg["target_demos"]
    max_attempts = target * MAX_ATTEMPT_MULTIPLIER

    print(f"\n{'='*70}")
    print(f"  COLLECTING: {scenario_name}")
    print(f"  Target: {target} successful demos | Max attempts: {max_attempts}")
    print(f"{'='*70}")

    env = suite.make(**cfg["env_kwargs"])
    policy = cfg["policy_class"]()

    # Open HDF5 file
    hdf5_path = os.path.join(OUTPUT_DIR, f"{scenario_name.lower()}_demos.hdf5")
    h5_file = h5py.File(hdf5_path, "w")
    data_grp = h5_file.create_group("data")
    data_grp.attrs["env"] = scenario_name
    data_grp.attrs["policy"] = "scripted"

    successes = 0
    attempts = 0
    start_time = time.time()

    for ep in range(max_attempts):
        if successes >= target:
            break

        np.random.seed(SEED_START + ep)
        result = run_episode(
            env, policy, scenario_name,
            max_steps=cfg["max_steps"],
            record_video=False,
        )
        attempts += 1

        if result["success"]:
            # Save to HDF5
            demo_grp = data_grp.create_group(f"demo_{successes}")
            demo_grp.attrs["seed"] = SEED_START + ep
            demo_grp.attrs["num_samples"] = len(result["actions"])
            demo_grp.attrs["total_reward"] = result["total_reward"]

            demo_grp.create_dataset("actions", data=np.array(result["actions"]))

            # Save observations
            obs_grp = demo_grp.create_group("obs")
            if result["observations"]:
                sample_keys = result["observations"][0].keys()
                for key in sample_keys:
                    obs_data = np.array([o[key] for o in result["observations"]])
                    obs_grp.create_dataset(key, data=obs_data)

            h5_file.flush()
            successes += 1

        # Progress update every 10 attempts
        if attempts % 10 == 0 or successes >= target:
            elapsed = time.time() - start_time
            rate = successes / max(attempts, 1)
            print(f"  [{attempts:4d} attempts]  {successes}/{target} demos collected  "
                  f"(success rate: {rate:.1%})  elapsed: {elapsed:.0f}s")

    env.close()

    # Finalize HDF5
    data_grp.attrs["total_demos"] = successes
    data_grp.attrs["total_attempts"] = attempts
    data_grp.attrs["success_rate"] = successes / max(attempts, 1)
    h5_file.close()

    elapsed = time.time() - start_time
    collection_summary[scenario_name] = {
        "successes": successes,
        "target": target,
        "attempts": attempts,
        "rate": successes / max(attempts, 1),
        "hdf5_path": hdf5_path,
        "elapsed_s": elapsed,
    }

    status = "DONE" if successes >= target else "INCOMPLETE"
    print(f"\n  {status}: {successes}/{target} demos in {attempts} attempts ({elapsed:.0f}s)")
    print(f"  Saved to: {hdf5_path}")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print(f"\n\n{'='*70}")
print(f"  DATA COLLECTION SUMMARY")
print(f"{'='*70}")
print(f"  {'Scenario':<25s} {'Collected':>10s} {'Target':>8s} {'Attempts':>10s} {'Rate':>8s} {'Time':>8s}")
print(f"  {'-'*70}")

all_complete = True
for name, r in collection_summary.items():
    done = "Y" if r["successes"] >= r["target"] else "N"
    if r["successes"] < r["target"]:
        all_complete = False
    print(f"  {name:<25s} {r['successes']:>5d}/{r['target']:<4d}  "
          f"{r['target']:>7d}  {r['attempts']:>9d}  {r['rate']:>7.1%}  {r['elapsed_s']:>6.0f}s")

print(f"\n  {'ALL TARGETS MET' if all_complete else 'SOME TARGETS NOT MET'}")
print(f"\n  HDF5 files saved in: {OUTPUT_DIR}/")
for name, r in collection_summary.items():
    print(f"    {r['hdf5_path']}  ({r['successes']} demos)")

In [ ]:
# Verify HDF5 files
print("\nHDF5 File Verification:")
print("=" * 60)

for name, r in collection_summary.items():
    path = r["hdf5_path"]
    if os.path.exists(path):
        with h5py.File(path, "r") as f:
            n_demos = f["data"].attrs.get("total_demos", 0)
            n_attempts = f["data"].attrs.get("total_attempts", 0)
            demo_keys = [k for k in f["data"].keys() if k.startswith("demo_")]
            print(f"\n  {name}:")
            print(f"    File: {path}")
            print(f"    Demos: {len(demo_keys)} (attr says {n_demos})")
            if demo_keys:
                d0 = f["data"][demo_keys[0]]
                print(f"    Demo 0: {d0.attrs.get('num_samples', '?')} steps")
                print(f"    Action shape: {d0['actions'].shape}")
                if "obs" in d0:
                    print(f"    Obs keys: {list(d0['obs'].keys())}")
    else:
        print(f"\n  {name}: FILE NOT FOUND at {path}")

---
## 7. (Optional) Save to Google Drive

In [ ]:
# Uncomment to save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# drive_dir = "/content/drive/MyDrive/robosuite_demos"
# os.makedirs(drive_dir, exist_ok=True)
#
# # Copy HDF5 files
# for name, r in collection_summary.items():
#     src = r["hdf5_path"]
#     dst = os.path.join(drive_dir, os.path.basename(src))
#     shutil.copy2(src, dst)
#     print(f"Copied {src} -> {dst}")
#
# # Copy trial videos
# shutil.copytree("trial_videos", os.path.join(drive_dir, "trial_videos"), dirs_exist_ok=True)
# print(f"\nAll files saved to {drive_dir}")